# ubctgdb: every command

Run this notebook from top to bottom with the R2 version of `ubctgdb` installed.
It creates two tiny example tables under a unique `example/` group in your configured
bucket. It does not modify the universe datasets. Upload cells need Object Read & Write access.

From the repository directory, install with `pip install -e .` in your notebook's Python environment.
Put the following four settings in the repository's `.env` (or another parent of the notebook's working directory):

```dotenv
R2_ENDPOINT_URL=https://YOUR_ACCOUNT_ID.r2.cloudflarestorage.com
R2_ACCESS_KEY_ID=your_access_key
R2_SECRET_ACCESS_KEY=your_secret_key
R2_BUCKET=ubctg-data
```

An **R2 Account API token** works: use the Access Key ID and Secret Access Key supplied
when you create it. No separate `token` or old `DB_*` settings are needed.
[Cloudflare's instructions](https://developers.cloudflare.com/r2/api/tokens/).
Keep credentials in `.env`, never in notebook cells or outputs.

Saved outputs show an executed example; run the notebook to create your own demo.

## Setup
Create two rows and unique demo names. Local downloads use a temporary directory.

In [1]:
import tempfile
from pathlib import Path
from uuid import uuid4
import pandas as pd
import ubctgdb as db

# Unique names keep this walkthrough separate from the club's real datasets.
group = "example/" + uuid4().hex[:12]
table = group + "/prices"
copy_table = group + "/prices_copy"
local_files = tempfile.TemporaryDirectory(prefix="ubctgdb-example-")
folder = Path(local_files.name)
prices = pd.DataFrame({"symbol": ["AAA", "BBB"], "price": [10.5, 20.0]})
print(prices.to_string(index=False))

symbol  price
   AAA   10.5
   BBB   20.0


## upload_dataframe()
Create a shared table from pandas. Returns a summary dictionary.

In [2]:
receipt = db.upload_dataframe(prices, table=table, description="Example prices")
print({key: receipt[key] for key in ["rows", "columns", "description"]})

{'rows': 2, 'columns': 2, 'description': 'Example prices'}


## list_tables()
List tables, newest additions first. This example filters to your unique demo group.

In [3]:
tables = db.list_tables(search=group)
print(tables[["table", "rows", "columns", "description"]].to_string(index=False))

                      table  rows  columns    description
example/77d9c47ff809/prices     2        2 Example prices


## describe()
Inspect summary information and column types without downloading the whole file.

In [4]:
info = db.describe(table)
print({key: info[key] for key in ["rows", "columns", "schema"]})

{'rows': 2, 'columns': 2, 'schema': {'symbol': 'large_string', 'price': 'double'}}


## preview()
Display the first rows using Parquet range reads.

In [5]:
sample = db.preview(table, rows=2)
print(sample.to_string(index=False))

symbol  price
   AAA   10.5
   BBB   20.0


## read_table()
Download the current table into pandas. You can select columns to load into memory.

In [6]:
df = db.read_table(table)
print(df.to_string(index=False))
print("Selected columns:")
print(db.read_table(table, columns=["symbol"]).to_string(index=False))

symbol  price
   AAA   10.5
   BBB   20.0
Selected columns:
symbol
   AAA
   BBB


## download_table()
Save Parquet locally without loading all rows into pandas. Returns a Path.

In [7]:
path = db.download_table(table, folder / "prices.parquet", overwrite=True)
print(path.name, "exists:", path.exists())

prices.parquet exists: True


## upload_parquet()
Publish an existing Parquet file directly, without a pandas conversion.

In [8]:
receipt = db.upload_parquet(path, table=copy_table, description="Example copy")
print({key: receipt[key] for key in ["rows", "columns", "description"]})

{'rows': 2, 'columns': 2, 'description': 'Example copy'}


## Replace your demo table
Existing remote names are protected by default. Use `replace_table=True` intentionally.
There is no history or undo. Re-run Setup to start a fresh demo; re-running upload cells
alone will otherwise raise FileExistsError.

In [9]:
updated = prices.copy()
updated.loc[0, "price"] = 11.0
db.upload_dataframe(updated, table=table, replace_table=True)
print(db.preview(table, rows=2).to_string(index=False))

symbol  price
   AAA   11.0
   BBB   20.0


## Finish
The two demo tables remain in R2 when you run this notebook. You may remove their
`tables/example/<your-run-id>/` objects in the Cloudflare dashboard when finished.
The next cell prints their exact names and removes only this notebook's local temporary files.

Useful options: `db.list_tables(sort_by="updated_at", ascending=False)`;
`db.list_tables(search="universe")`. Listing also includes creation/update dates and byte sizes.
Each read downloads fresh data; there is no persistent cache. Column selection reduces
pandas memory use, not download size. Previews can transfer substantial data for large row groups.

In [10]:
print("Demo tables:", table, copy_table, sep="\n")
local_files.cleanup()

Demo tables:
example/77d9c47ff809/prices
example/77d9c47ff809/prices_copy
